# ML Classification Assignment

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)
(the same data is bundled with scikit-learn as `load_breast_cancer`)

**Task:** Binary classification — Malignant (0) vs Benign (1)

**Models:** Logistic Regression, Decision Tree, K-Nearest Neighbors, Gaussian Naive Bayes, Support Vector Machine

**Metrics:** Accuracy, AUC, Precision, Recall, F1 Score, Matthews Correlation Coefficient (MCC)

## Imports

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
)

RANDOM_STATE = 42

## Step 1: Dataset Choice

**Why this dataset?**
- **569 instances** (≥ 500 required) and **30 numeric features** (≥ 12 required)
- No missing values, all features numeric — no imputation or encoding needed
- Moderate class imbalance (212 malignant / 357 benign), which makes MCC and AUC meaningful alongside accuracy

The 30 features are computed from digitized images of a fine needle aspirate (FNA) of a breast mass: for each of 10 cell-nucleus characteristics (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension) the **mean**, **standard error** and **worst** (largest) value are recorded.

In [2]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print(f"Instances      : {X.shape[0]}")
print(f"Features       : {X.shape[1]}")
print(f"Missing values : {int(X.isna().sum().sum())}")
print(f"Classes        : {dict(enumerate(data.target_names))}")
X.head()

Instances      : 569
Features       : 30
Missing values : 0
Classes        : {0: np.str_('malignant'), 1: np.str_('benign')}


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# Class distribution
class_counts = y.value_counts().sort_index().rename(index=dict(enumerate(data.target_names)))
pd.DataFrame({"count": class_counts, "percent": (class_counts / len(y) * 100).round(2)})

,count,percent
target,,
malignant,212,37.26
benign,357,62.74


In [4]:
# Summary statistics of the features
X.describe().T

,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


### Train / Test Split
A stratified 80/20 split keeps the malignant/benign ratio identical in the training and test sets.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train size: {len(X_train)}   Test size: {len(X_test)}")

Train size: 455   Test size: 114


## Step 2: Classification Models and Evaluation Metrics

### Evaluation helper
For each model we fit on the training set and compute all six metrics on the test set.
- **AUC** needs a continuous score: we use the predicted probability of class 1 where available, otherwise the SVM's signed distance to the decision boundary (`decision_function`).
- **Precision / Recall / F1** are reported for the positive class (1 = benign).
- **MCC** ranges from −1 to +1 and is robust to class imbalance.

In [6]:
def evaluate(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }

    print(f"=== {name} ===")
    for k, v in metrics.items():
        print(f"{k:<10}: {v:.4f}")
    print("\nConfusion matrix (rows = actual, cols = predicted):")
    print(pd.DataFrame(
        confusion_matrix(y_test, y_pred),
        index=[f"actual {c}" for c in data.target_names],
        columns=[f"pred {c}" for c in data.target_names],
    ))
    print("\n" + classification_report(y_test, y_pred, target_names=data.target_names, digits=4))
    return metrics

results = {}

### Model 1: Logistic Regression
A linear model; features are standardized inside a `Pipeline` so the scaler is fit on training data only (no data leakage).

In [7]:
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
results["Logistic Regression"] = evaluate("Logistic Regression", lr)

=== Logistic Regression ===
Accuracy  : 0.9825
AUC       : 0.9954
Precision : 0.9861
Recall    : 0.9861
F1        : 0.9861
MCC       : 0.9623

Confusion matrix (rows = actual, cols = predicted):
                  pred malignant  pred benign
actual malignant              41            1
actual benign                  1           71

              precision    recall  f1-score   support

   malignant     0.9762    0.9762    0.9762        42
      benign     0.9861    0.9861    0.9861        72

    accuracy                         0.9825       114
   macro avg     0.9812    0.9812    0.9812       114
weighted avg     0.9825    0.9825    0.9825       114



### Model 2: Decision Tree Classifier
Tree models are scale-invariant, so no scaling is applied. `max_depth=5` limits overfitting.

In [8]:
dt = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
results["Decision Tree"] = evaluate("Decision Tree", dt)

=== Decision Tree ===
Accuracy  : 0.9211
AUC       : 0.9163
Precision : 0.9565
Recall    : 0.9167
F1        : 0.9362
MCC       : 0.8341

Confusion matrix (rows = actual, cols = predicted):
                  pred malignant  pred benign
actual malignant              39            3
actual benign                  6           66

              precision    recall  f1-score   support

   malignant     0.8667    0.9286    0.8966        42
      benign     0.9565    0.9167    0.9362        72

    accuracy                         0.9211       114
   macro avg     0.9116    0.9226    0.9164       114
weighted avg     0.9234    0.9211    0.9216       114



### Model 3: K-Nearest Neighbor Classifier
KNN is distance-based, so standardization is essential. `k = 5`.

In [9]:
knn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_neighbors=5)),
])
results["K-Nearest Neighbors"] = evaluate("K-Nearest Neighbors", knn)

=== K-Nearest Neighbors ===
Accuracy  : 0.9561
AUC       : 0.9788
Precision : 0.9589
Recall    : 0.9722
F1        : 0.9655
MCC       : 0.9054

Confusion matrix (rows = actual, cols = predicted):
                  pred malignant  pred benign
actual malignant              39            3
actual benign                  2           70

              precision    recall  f1-score   support

   malignant     0.9512    0.9286    0.9398        42
      benign     0.9589    0.9722    0.9655        72

    accuracy                         0.9561       114
   macro avg     0.9551    0.9504    0.9526       114
weighted avg     0.9561    0.9561    0.9560       114



### Model 4: Naive Bayes Classifier (Gaussian)
Gaussian NB is chosen because all features are continuous (Multinomial NB is meant for count data).

In [10]:
nb = GaussianNB()
results["Naive Bayes (Gaussian)"] = evaluate("Naive Bayes (Gaussian)", nb)

=== Naive Bayes (Gaussian) ===
Accuracy  : 0.9386
AUC       : 0.9878
Precision : 0.9452
Recall    : 0.9583
F1        : 0.9517
MCC       : 0.8676

Confusion matrix (rows = actual, cols = predicted):
                  pred malignant  pred benign
actual malignant              38            4
actual benign                  3           69

              precision    recall  f1-score   support

   malignant     0.9268    0.9048    0.9157        42
      benign     0.9452    0.9583    0.9517        72

    accuracy                         0.9386       114
   macro avg     0.9360    0.9315    0.9337       114
weighted avg     0.9384    0.9386    0.9384       114



### Model 5: Support Vector Machine
RBF-kernel SVM with standardized features.

In [11]:
svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", random_state=RANDOM_STATE)),
])
results["Support Vector Machine"] = evaluate("Support Vector Machine", svm)

=== Support Vector Machine ===
Accuracy  : 0.9825
AUC       : 0.9950
Precision : 0.9861
Recall    : 0.9861
F1        : 0.9861
MCC       : 0.9623

Confusion matrix (rows = actual, cols = predicted):
                  pred malignant  pred benign
actual malignant              41            1
actual benign                  1           71

              precision    recall  f1-score   support

   malignant     0.9762    0.9762    0.9762        42
      benign     0.9861    0.9861    0.9861        72

    accuracy                         0.9825       114
   macro avg     0.9812    0.9812    0.9812       114
weighted avg     0.9825    0.9825    0.9825       114



## Comparison of All Models

In [12]:
results_df = pd.DataFrame(results).T.round(4)
results_df.index.name = "Model"
results_df.to_csv("model_comparison_results.csv")
results_df.style.highlight_max(axis=0, color="lightgreen").format("{:.4f}")

,Accuracy,AUC,Precision,Recall,F1,MCC
Model,,,,,,
Logistic Regression,0.9825,0.9954,0.9861,0.9861,0.9861,0.9623
Decision Tree,0.9211,0.9163,0.9565,0.9167,0.9362,0.8341
K-Nearest Neighbors,0.9561,0.9788,0.9589,0.9722,0.9655,0.9054
Naive Bayes (Gaussian),0.9386,0.9878,0.9452,0.9583,0.9517,0.8676
Support Vector Machine,0.9825,0.9950,0.9861,0.9861,0.9861,0.9623


In [13]:
best_mcc = results_df["MCC"].max()
best = results_df.index[results_df["MCC"] == best_mcc].tolist()
print(f"Best model(s) by MCC: {', '.join(best)} (MCC = {best_mcc:.4f})")

Best model(s) by MCC: Logistic Regression, Support Vector Machine (MCC = 0.9623)


## Export Trained Models
Each fitted model is saved to its own file in `models/` with `joblib` (scikit-learn's recommended serializer).
The scaled models are saved as full `Pipeline`s, so the `StandardScaler` is included and the loaded model can be used directly on raw feature values.

In [14]:
import os
import joblib

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

trained_models = {
    "logistic_regression": lr,
    "decision_tree": dt,
    "knn": knn,
    "naive_bayes": nb,
    "svm": svm,
}

for file_name, model in trained_models.items():
    path = os.path.join(MODEL_DIR, f"{file_name}.pkl")
    joblib.dump(model, path)
    print(f"Saved {path:<35} ({os.path.getsize(path) / 1024:.1f} KB)")

# Held-out test split (features + target) for evaluating models in the Streamlit app.
X_test.assign(target=y_test).to_csv("test_data.csv", index=False)
print("Saved test_data.csv")

Saved models/logistic_regression.pkl      (3.0 KB)
Saved models/decision_tree.pkl            (4.3 KB)
Saved models/knn.pkl                      (112.9 KB)
Saved models/naive_bayes.pkl              (2.5 KB)
Saved models/svm.pkl                      (28.2 KB)
Saved test_data.csv


### Verify the exported models
Reload every model from disk and confirm it makes exactly the same predictions on the test set as the in-memory model.

In [15]:
for file_name, model in trained_models.items():
    loaded = joblib.load(os.path.join(MODEL_DIR, f"{file_name}.pkl"))
    same = (loaded.predict(X_test) == model.predict(X_test)).all()
    acc = accuracy_score(y_test, loaded.predict(X_test))
    print(f"{file_name:<20} reloaded OK | identical predictions: {same} | test accuracy: {acc:.4f}")

logistic_regression  reloaded OK | identical predictions: True | test accuracy: 0.9825
decision_tree        reloaded OK | identical predictions: True | test accuracy: 0.9211
knn                  reloaded OK | identical predictions: True | test accuracy: 0.9561
naive_bayes          reloaded OK | identical predictions: True | test accuracy: 0.9386
svm                  reloaded OK | identical predictions: True | test accuracy: 0.9825


### Using an exported model
```python
import joblib
from sklearn.datasets import load_breast_cancer

model = joblib.load("models/logistic_regression.pkl")
X = load_breast_cancer(as_frame=True).data
model.predict(X.head())   # 0 = malignant, 1 = benign
```
Input must have the same 30 features in the same order as training.

## Observations

| Model | Observation |
|---|---|
| **Logistic Regression** | Tied for best on every metric (MCC ≈ 0.96, AUC ≈ 0.995). After standardization the classes are nearly linearly separable, so a simple linear boundary works extremely well. |
| **Decision Tree** | Weakest model on all metrics (MCC ≈ 0.83). Axis-aligned splits struggle with this smooth, correlated feature space, and a single tree has high variance. |
| **K-Nearest Neighbors** | Strong (MCC ≈ 0.91) thanks to scaling; slightly noisier than LR/SVM near the class boundary. |
| **Naive Bayes (Gaussian)** | High AUC (≈ 0.99) — it ranks samples well — but lower MCC (≈ 0.87) because many features are highly correlated (e.g. radius/perimeter/area), violating the independence assumption. |
| **Support Vector Machine** | Tied with Logistic Regression for best accuracy, precision, recall, F1 and MCC; AUC ≈ 0.995. The RBF kernel captures the boundary well with scaled features. |

**Conclusion:** Logistic Regression and SVM are the best performers on this dataset (only 2 misclassifications out of 114 test samples each). Logistic Regression is preferable in practice because it is simpler and more interpretable.